In [ ]:
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT
from whoosh.qparser import QueryParser
import os

# ---------------------
# 1. Load documents
# ---------------------
documents = [
    {"text": "Python is a programming language."},
    {"text": "FAISS is used for vector similarity search."},
    {"text": "LangChain is a framework for building RAG applications."}
]

# ---------------------
# 2. Embedding model
# ---------------------
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
def embed_text(texts):
    return embedding_model.encode(texts, convert_to_tensor=True)

# ---------------------
# 3. Vector DB
# ---------------------
texts = [doc["text"] for doc in documents]
embeddings = embed_text(texts)
vector_db = FAISS.from_embeddings(texts, embeddings)

# ---------------------
# 4. Sparse Search with Whoosh
# ---------------------
schema = Schema(content=TEXT(stored=True))
if not os.path.exists("indexdir"):
    os.mkdir("indexdir")
ix = create_in("indexdir", schema)
writer = ix.writer()
for doc in documents:
    writer.add_document(content=doc["text"])
writer.commit()

def sparse_search(query):
    with ix.searcher() as searcher:
        qp = QueryParser("content", schema=ix.schema)
        q = qp.parse(query)
        results = searcher.search(q, limit=3)
        return [r['content'] for r in results]

# ---------------------
# 5. LLM Generator
# ---------------------
model_name = "TheBloke/MPT-7B-Chat-GGML"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=512)
llm = HuggingFacePipeline(pipeline=pipe)

# ---------------------
# 6. Hybrid Retrieval Function
# ---------------------
def hybrid_rag(query):
    # Dense retrieval
    dense_docs = vector_db.similarity_search(query, k=2)
    dense_texts = [d.page_content for d in dense_docs]
    
    # Sparse retrieval
    sparse_texts = sparse_search(query)
    
    # Combine
    combined_context = "\n".join(dense_texts + sparse_texts)
    
    # LLM generation
    prompt = f"Answer the question using the context below:\n{combined_context}\nQuestion: {query}\nAnswer:"
    response = llm(prompt)[0]['generated_text']
    return response

# ---------------------
# 7. Test
# ---------------------
query = "What is FAISS?"
answer = hybrid_rag(query)
print(answer)